In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import pickle

sys.path.append(os.path.abspath("../analysis_tools/"))
from utils import * 

# load first few rows from parquet
import fastparquet

# for median absolute deviatio
import scipy.stats as ss
from scipy.special import logit

# for Louvain clustering
import community as community_louvain
import networkx as nx
from sklearn.neighbors import NearestNeighbors
import scipy.sparse as sp
import scipy.spatial as spt

# make cell crops
import skimage.io

# for legends heatmap annotations
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker

# upset plots
from upsetplot import generate_counts, plot
from upsetplot import UpSet


In [ ]:
# Import other datasets
# Anderson data
anderson_data = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/anderson_2021_variantdata_Formatted.csv')

# Clinvar
clinvar_variants_filterconditions = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/clinvar_LMNA_missense_ex345_061824_fixed.csv')
condition_map = {'DCM':'DCM', 'Myopathy':'EDMD', 'Lipodystrophy':'FPLD', 'Progeria':'HGPS'}
for disease in ['DCM','Myopathy','Lipodystrophy','Progeria']:
    clinvar_variants_filterconditions[disease] = \
        clinvar_variants_filterconditions["Mapped Conditions"]\
            .map(lambda x: True if condition_map[disease] in x else False)

# Define one way to color variants, use when highlighting specific variants
variant_type_palette = \
    {'Missense':'grey',
     'Synonymous':'darkgreen',
     'WT':'grey', 
     'Frameshift':'purple', 
     '3nt Deletion':'grey', 
     'Nonsense':'purple', 
     'Other':'grey'}
mutation_types=['Missense','Synonymous','Frameshift']

# Variants filtered for spliceAI scores
lmna_variants_spliceAIfiltered = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/LMNA_AM_CV.csv')
lmna_variants_spliceAIfiltered = \
    lmna_variants_spliceAIfiltered.rename(columns={'protein_variant':'Variant'})\
    [['Variant','am_pathogenicity']]
lmna_variants_spliceAIfiltered = (
    lmna_variants_spliceAIfiltered
    .groupby('Variant', as_index=False)
    .agg({'am_pathogenicity': 'mean'})
)


In [ ]:
# Import Gwen's ROC AUC
roc_auc_lmna = pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/AUROC/results/train_results.csv')
roc_auc_lmna = roc_auc_lmna[['aaChanges','test_roc_auc','Example Count']].rename(columns={'aaChanges':'Variant','test_roc_auc':'AUC ROC'})


In [ ]:
# EVE variants
lmna_variants_EVE = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/LMNA_HUMAN.csv')
lmna_variants_EVE = lmna_variants_EVE[['wt_aa','position','mt_aa','EVE_scores_ASM','uncertainty_ASM','EVE_classes_75_pct_retained_ASM']]
lmna_variants_EVE = lmna_variants_EVE.dropna(axis=0)
lmna_variants_EVE['Variant'] = \
    lmna_variants_EVE.apply(lambda row: row['wt_aa']+str(row['position']) + row['mt_aa'], axis=1)
lmna_variants_EVE = lmna_variants_EVE[['Variant','EVE_scores_ASM','uncertainty_ASM','EVE_classes_75_pct_retained_ASM']]


In [ ]:
# REVEL variants
lmna_variants_REVEL = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/REVEL_LMNA.csv')
lmna_variants_REVEL = \
    lmna_variants_REVEL[lmna_variants_REVEL['tag'].isin(['CCDS'])]

# 2. Load the GTF for GRCh38 (replace with your GTF filename)
gtf = pd.read_csv(
    './Homo_sapiens.GRCh38.104.gtf.gz', #https://ftp.ensembl.org/pub/release-104/gtf/homo_sapiens/Homo_sapiens.GRCh38.104.gtf.gz
    sep='\t',
    comment='#',
    header=None,
    names=['chrom','source','feature','start','end','score','strand','frame','attribute']
)

# 3. Filter to the CDS exons of the MANE Select transcript
transcript_id = 'ENST00000368300'
mask = (
    (gtf['feature'] == 'CDS') &
    (gtf['attribute'].str.contains(f'transcript_id "{transcript_id}"')) &
    (gtf['chrom'].isin([1,'1','chr1']))
)
cds = gtf[mask].copy()

# 4. Order exons in transcript order
strand = cds['strand'].iat[0]
cds = cds.sort_values('start', ascending=(strand == '+'))
intervals = cds[['start','end']].values.tolist()

# 5. Map a genomic position to its 1‑based protein residue number
def genomic_to_residue(pos):
    cds_pos = 0
    for start, end in intervals:
        if pos < start:
            continue
        if pos > end:
            cds_pos += (end - start + 1)
            continue
        # pos is within this exon
        cds_pos += (pos - start + 1)
        return ((cds_pos - 1) // 3) + 1
    return None

# 6. Apply to get the residue number, dropping rows where it is NaN
lmna_variants_REVEL['aa_pos'] = pd.to_numeric(lmna_variants_REVEL['grch38_pos'].apply(genomic_to_residue), errors='coerce')
lmna_variants_REVEL = lmna_variants_REVEL.dropna(subset=['aa_pos']).copy()

# 7. Build the variant label
lmna_variants_REVEL['Variant'] = lmna_variants_REVEL['aaref'] + lmna_variants_REVEL['aa_pos'].astype(int).astype(str) + lmna_variants_REVEL['aaalt']
lmna_variants_REVEL = lmna_variants_REVEL[['Variant','REVEL']]


In [6]:
# LMNA information
start_pos = 178
end_pos = 273
LMNA_domain_map = {}
for pos in range(start_pos, end_pos+1):
    if pos < 222:
        LMNA_domain_map[pos] = "Coil 1B"
    elif pos <= 241:
        LMNA_domain_map[pos] = "Linker 12"
    else:
        LMNA_domain_map[pos] = "Coil 2A"
LMNA_domain_colors = \
    {
        'Coil 1B':'goldenrod',
        'Linker 12':'lime',
        'Coil 2A':'deepskyblue'
    }

# Set up for making feature plots
LMNA_WT_nuc_seq = \
    'ATGGAGACCCCGTCCCAGCGGCGCGCCACCCGCAGCGGGGCGCAGGCCAGCTCCACTCCGCTGTCGCCCACCCGCATCACCCGGCTGCAGGAGAAGGAGGACCTGCAGGAGCTCAATGATCGCTTGGCGGTCTACATCGACCGTGTGCGCTCGCTGGAAACGGAGAACGCAGGGCTGCGCCTTCGCATCACCGAGTCTGAAGAGGTGGTCAGCCGCGAGGTGTCCGGCATCAAGGCCGCCTACGAGGCCGAGCTCGGGGATGCCCGCAAGACCCTTGACTCAGTAGCCAAGGAGCGCGCCCGCCTGCAGCTGGAGCTGAGCAAAGTGCGTGAGGAGTTTAAGGAGCTGAAAGCGCGCAATACCAAGAAGGAGGGTGACCTGATAGCTGCTCAGGCTCGGCTGAAGGACCTGGAGGCTCTGCTGAACTCCAAGGAGGCCGCACTGAGCACTGCTCTCAGTGAGAAGCGCACGCTGGAGGGCGAGCTGCATGATCTGCGGGGCCAGGTGGCCAAGCTTGAGGCAGCCCTAGGTGAGGCCAAGAAACAGCTGCAGGACGAAATGCTGCGGCGGGTGGATGCTGAGAACAGGCTGCAGACCATGAAGGAGGAACTGGACTTCCAGAAGAACATCTACAGTGAGGAGCTGCGTGAGACCAAGCGCCGTCATGAGACCCGACTGGTGGAGATTGACAATGGGAAGCAGCGTGAGTTTGAGAGCCGGCTGGCGGATGCGCTGCAGGAACTGCGGGCCCAGCATGAGGACCAGGTGGAGCAGTATAAGAAGGAGCTGGAGAAGACTTATTCTGCCAAGCTGGACAATGCCAGGCAGTCTGCTGAGAGGAACAGCAACCTGGTGGGGGCTGCCCACGAGGAGCTGCAGCAGTCGCGCATCCGCATCGACAGCCTCTCTGCCCAGCTCAGCCAGCTCCAGAAGCAGCTGGCAGCCAAGGAGGCGAAGCTTCGAGACCTGGAGGACTCACTGGCCCGTGAGCGGGACACCAGCCGGCGGCTGCTGGCGGAAAAGGAGCGGGAGATGGCCGAGATGCGGGCAAGGATGCAGCAGCAGCTGGACGAGTACCAGGAGCTTCTGGACATCAAGCTGGCCCTGGACATGGAGATCCACGCCTACCGCAAGCTCTTGGAGGGCGAGGAGGAGAGGCTACGCCTGTCCCCCAGCCCTACCTCGCAGCGCAGCCGTGGCCGTGCTTCCTCTCACTCATCCCAGACACAGGGTGGGGGCAGCGTCACCAAAAAGCGCAAACTGGAGTCCACTGAGAGCCGCAGCAGCTTCTCACAGCACGCACGCACTAGCGGGCGCGTGGCCGTGGAGGAGGTGGATGAGGAGGGCAAGTTTGTCCGGCTGCGCAACAAGTCCAATGAGGACCAGTCCATGGGCAATTGGCAGAtCAAGCGCCAGAATGGAGACGACCCACTGCTCACCTACCGGTTCCCACCAAAGTTCACCCTGAAGGCTGGGCAGGTGGTGACGATCTGGGCTGCAGGAGCTGGGGCCACCCACAGCCCCCCTACCGACCTGGTGTGGAAGGCACAGAACACCTGGGGCTGCGGGAACAGCCTGCGTACGGCTCtCATCAACTCCACTGGGGAAGAAGTGGCCATGCGCAAGCTGGTGCGCTCAGTGACTGTGGTTGAGGACGACGAGGATGAGGATGGAGATGACCTGCTCCATCACCACCACGGCTCCCACTGCAGCAGCTCGGGGGACCCCGCTGAGTACAACCTGCGCTCGCGCACCGTGCTGTGCGGGACCTGCGGGCAGCCTGCCGACAAGGCATCTGCCAGCGGCTCAGGAGCCCAGGTGGGCGGACCCATCTCCTCTGGCTCTTCTGCCTCCAGTGTCACGGTCACTCGCAGCTACCGCAGTGTGGGGGGCAGTGGGGGTGGCAGCTTCGGGGACAATCTGGTCACCCGCTCCTACCTCCTGGGCAACTCCAGCCCCCGAACCCAGAGCCCCCAGAACTGCAGCATCATGTAG'
LMNA_WT_aa_seq = \
    str(Seq(LMNA_WT_nuc_seq).translate())
lmna_pymol_view = \
    '''set_view (\
        -0.915147185,    0.403044611,   -0.007183676,\
        -0.175614133,   -0.414666086,   -0.892865419,\
        -0.362844229,   -0.815845907,    0.450264663,\
         0.000000000,    0.000000000, -382.015167236,\
      -190.245574951, -108.335136414,  109.851905823,\
       301.183654785,  462.846679688,  -20.000000000 )'''
lmna_pymol_view2 = \
    '''set_view (\
     0.071904697,   -0.778118074,   -0.623987436,\
     0.870689154,    0.354146332,   -0.341291487,\
     0.486548126,   -0.518758714,    0.702964723,\
     0.000000000,    0.000000000, -374.783477783,\
    -2.406383991,  -20.413566589,  -11.670532227,\
   295.482147217,  454.084808350,  -20.000000000 )'''


In [ ]:
# Read profiles
df_profiles_merged = pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_averaged_medianplusEMD_010425.csv', index_col=None)
df_profiles_merged.drop(columns='Unnamed: 0',inplace=True)
df_profiles_merged['Variant_Class'] = \
    pd.Categorical(df_profiles_merged['Variant'].astype(str).apply(variant_classification),
                    categories=mutation_types, ordered=True)
features_common = [c for c in df_profiles_merged.columns if c not in ['Variant','Variant_Class',
                                                                      'NCells_R1','NCells_R2',
                                                                      'UMAP1','UMAP2',
                                                                      'Louvain Cluster']]
# Relabel clusters
cluster_remap = \
    {
        0: 8,
        1: 9,
        2: 2,
        3: 10,
        4: 3,
        5: 4,
        6: 6,
        7: 7,
        8: 5,
        9: 1
    }
df_profiles_merged['Cluster'] = \
    df_profiles_merged['Louvain Cluster'].map(lambda x: cluster_remap[x])

# Read p-values
df_KSpvalues_merged = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_merged_KSpvalues_010924.csv')
df_KSpvalues_merged.index = df_KSpvalues_merged['Variant']
df_KSpvalues_merged.drop(columns='Variant',inplace=True)

# Read features
variant_medians_merged = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_merged_featuremedians_010625.csv')
variant_medians_merged.drop(columns='Unnamed: 0', inplace=True)
variant_medians_merged['Variant_Class'] = \
    pd.Categorical(variant_medians_merged['Variant'].astype(str).apply(variant_classification),
                    categories=mutation_types, ordered=True)

variant_EMD_merged = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_merged_featureEMD_010925.csv')
variant_EMD_merged.drop(columns='Unnamed: 0', inplace=True)
variant_EMD_merged['Variant_Class'] = \
   pd.Categorical(variant_EMD_merged['Variant'].astype(str).apply(variant_classification),
                    categories=mutation_types, ordered=True)

# Get umap coordinates
umap_coords = \
    df_profiles_merged[['Variant','UMAP1','UMAP2']]

# Highlight biochemical controls
punctuate_controls = ['N195K','L248P','E203G','R189P']
non_punctuate_controls = ['D192V', 'R189W', 'E203K', 'E203V', 'E262K', 'E202K', 'R190Q']
total_controls = punctuate_controls + non_punctuate_controls
umap_coords['Highlight'] = \
    umap_coords['Variant'].map(lambda x: True if x in total_controls else False)
umap_coords['HighlightClass'] = \
    umap_coords['Variant'].map(lambda x: 'Punctuate' if x in punctuate_controls \
                                               else 'Non-Punctuate' if x in non_punctuate_controls \
                                               else 'WT' if x == 'WT' \
                                                else np.nan)


In [8]:
# Plot impact as a function of aa to
aliphatic       = ["A", "V", "I", "L"]
aromatic        = ["F", "Y", "W"]
nonpolar        = ["C", "M"]
loop_forming    = ["G", "P"]
polar           = ["S", "T", "N", "Q"]
acidic          = ["D", "E"]
basic           = ["H", "K", "R"]
aa_type = aliphatic+aromatic+nonpolar+loop_forming+polar+acidic+basic
palette_aato={}
palette_aato['Syn']='green'
for t in aliphatic:
    palette_aato[t]='tan'
for t in aromatic:
    palette_aato[t]='darkorange'
for t in nonpolar:
    palette_aato[t]='gold'
for t in loop_forming:
    palette_aato[t]='mediumspringgreen'
for t in polar:
    palette_aato[t]='cyan'
for t in acidic:
    palette_aato[t]='maroon'
for t in basic:
    palette_aato[t]='cornflowerblue'

df_toplot = df_profiles_merged.query('Variant_Class in ["Synonymous","Missense"]').copy()
df_toplot['AA To'] = \
    df_toplot\
        .apply(lambda row: 'Syn' if row['Variant_Class']=='Synonymous' else row['Variant'][-1],axis=1)
fig, ax = plt.subplots(figsize=(6,4))
boxplot_with_significance(
    df=df_toplot,
    x_col='AA To',
    y_col='Morphological Impact Score',
    hue_col='AA To',  # same as x_col
    order=['Syn']+aa_type,
    alpha=0.001,    # Show significance if p < 0.001
    pairs=[("Syn",t) for t in aa_type],
    palette=palette_aato,
    offset=0.005,
    ax=ax,
    show_points=True,
    anno_sample_sizes=False
)

plt.ylim(0, 1)
plt.tight_layout()
plt.ylabel('Impact Score', fontsize=14)
plt.xlabel('')
plt.tick_params(labelsize=12)
plt.close()
fig.savefig('./consensus_plots_postreview/morphological_impact.aato.011325.pdf', dpi=600, bbox_inches='tight')


In [9]:
# Plot impact as a function of aa to
df_toplot = df_profiles_merged.query('Variant_Class in ["Synonymous","Missense"]').copy()
df_toplot['AA To'] = \
    df_toplot\
        .apply(lambda row: 'Syn' if row['Variant_Class']=='Synonymous' else row['Variant'][-1],axis=1)
df_toplot = \
    df_toplot.merge(roc_auc_lmna, on='Variant', how='inner')
fig, ax = plt.subplots(figsize=(6,4))
boxplot_with_significance(
    df=df_toplot,
    x_col='AA To',
    y_col='AUC ROC',
    hue_col='AA To',  # same as x_col
    order=['Syn']+aa_type,
    alpha=0.001,    # Show significance if p < 0.001
    pairs=[("Syn",t) for t in aa_type],
    palette=palette_aato,
    offset=0.005,
    ax=ax,
    show_points=True,
    anno_sample_sizes=False
)

plt.ylim(0.45, 1)
plt.tight_layout()
plt.ylabel('Distinguishability Score', fontsize=14)
plt.xlabel('')
plt.tick_params(labelsize=12)
plt.close()
fig.savefig('./consensus_plots_postreview/auc_roc.aato.052725.pdf', dpi=600, bbox_inches='tight')


In [10]:
# AA groupings
aliphatic       = ["A", "V", "I", "L"]
aromatic        = ["F", "Y", "W"]
nonpolar        = ["C", "M"]
loop_forming    = ["G", "P"]
polar           = ["S", "T", "N", "Q"]
acidic          = ["D", "E"]
basic           = ["H", "K", "R"]
aa_groups = {
    "A/V/I/L":    aliphatic,
    "F/Y/W":      aromatic,
    "C/M":        nonpolar,
    "G/P":        loop_forming,
    "S/T/N/Q":    polar,
    "D/E":        acidic,
    "H/K/R":      basic,
}
aa_group_order = list(aa_groups.keys())

# Palette for AA groups (tweak as desired)
group_palette = {
    "A/V/I/L":    "tan",
    "F/Y/W":      "orange",
    "C/M":        "gold",
    "G/P":        "mediumspringgreen",
    "S/T/N/Q":    "cyan",
    "D/E":        "maroon",
    "H/K/R":      "cornflowerblue",
}

# Missense only, parse aa_to and group
missense_toplot = (
    df_profiles_merged
    .query("Variant_Class == 'Missense'")
    .assign(
        aa_to=lambda d: d['Variant'].str[-1],
        Cluster=lambda d: d['Cluster'].astype(int)
    )
)
def map_group(aa):
    for g, aas in aa_groups.items():
        if aa in aas:
            return g
    return "other"

missense_toplot["aa_group"] = missense_toplot["aa_to"].map(map_group)
missense_toplot = missense_toplot.query("aa_group != 'other'")

# counts (not proportions) for the test
nclust = np.max(missense_toplot['Cluster'])
cluster_order = range(1,nclust+1)
counts_table = (
    missense_toplot
    .groupby(["Cluster", "aa_group"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=cluster_order, columns=aa_group_order)
)

# counts_table you already built is: index=Cluster, columns=aa_group
counts_aagroup = (
    counts_table
    .T
    .reindex(index=aa_group_order)     # keep AA group order
    .reindex(columns=cluster_order)    # keep cluster order
)

fig, ax = plt.subplots(figsize=(5, 3))
ax, frac, p, q, orr = plot_heatmap_fractional_bycluster(
    counts_aagroup,
    pseudocount=0,
    alpha=0.05,
    vmax=0.35,
    cmap="viridis",
    ax=ax,
    xlabel="",
    ylabel="Louvain cluster",
    normalize="within_group",
    cbar_label="Fraction variants in cluster",
    anno_sample_sizes=False
)

for lab in ax.get_xticklabels():
    lab.set_color(group_palette.get(lab.get_text(), "black"))

fig.savefig("./consensus_plots_postreview/LMNAT3.aagroup_bycluster.within_group.pdf",
            dpi=600, bbox_inches="tight")
plt.close()

In [11]:
# Combine all features
df_features_tot = \
    df_profiles_merged[['Variant',
                        'NCells_R1','NCells_R2',
                        'Variant_Class',
                        'UMAP1','UMAP2',
                        'Cluster',
                        'Morphological Impact Score']]\
        .merge(variant_medians_merged\
                   .drop(columns='Variant_Class')\
                   .rename(columns={c:c+'_median' for c in variant_medians_merged.columns if "Variant" not in c}),
               on='Variant'
              )\
        .merge(variant_EMD_merged\
                   .drop(columns='Variant_Class')\
                   .rename(columns={c:c+'_EMD' for c in variant_medians_merged.columns if "Variant" not in c}),
               on='Variant')


In [13]:
!mkdir consensus_plots_postreview/heatmap
!mkdir consensus_plots_postreview/distribution
!mkdir consensus_plots_postreview/umap
!mkdir consensus_plots_postreview/pdb

In [ ]:
# Look at specific features, adding domain annotation onto heatmaps
feature_to_heatmap = 'Morphological Impact Score'
feature_scores, pymol_command = \
    plot_feature(
        df_profiles_merged[['Variant',feature_to_heatmap,'Variant_Class']], # not in variant medians merged
        umap_coords,
        feature_to_heatmap,
        "LMNA Tile 3 Morphological Impact Score",
        "LMNA.T3.impactscore",
        dist_plot=True,
        umap_plot=True,
        heatmap_plot=True,
        pdb_plot=True,      
        highlight_variants=False,
        save_plots=True,
        stops=False,
        threentdels=False,
        syn_zscore=False,
        domain_map=LMNA_domain_map,
        domain_colors=LMNA_domain_colors,
        start_pos=start_pos,
        end_pos=end_pos,
        wt_nuc_seq=LMNA_WT_nuc_seq,
        wt_aa_seq=LMNA_WT_aa_seq,
        remove_nonclassified_variants=False,
        input_pdb='../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/6jlb.pdb',
        pymol_view=lmna_pymol_view,
        image_size=(5000,2500),
        dpi=1000,
        pymol_exec="$HOME/pymol/pymol -c",
        extra_pymol="show sticks\n",
        dis_path='./consensus_plots_postreview/distribution/',
        heatmap_path='./consensus_plots_postreview/heatmap/',
        umap_path='./consensus_plots_postreview/umap/',
        pdb_path='./consensus_plots_postreview/pdb/',
        heatmap_figure_size=(40,20),
        vmin=0.4,
        vcenter=0.65,
        vmax=0.9,
        heatmap_color_scheme='YlOrBr',
        show_plots=False)
#!{pymol_command}


In [ ]:
# Look at specific features, adding domain annotation onto heatmaps
feature_to_heatmap = 'AUC ROC'
feature_scores, pymol_command = \
    plot_feature(
        df_profiles_merged[['Variant','Variant_Class']]\
                .merge(roc_auc_lmna, on='Variant', how='left'), # not in variant medians merged
        umap_coords,
        feature_to_heatmap,
        "LMNA Tile 3 AUROC score",
        "LMNA.T3.AUROCscore",
        dist_plot=True,
        umap_plot=True,
        heatmap_plot=True,
        pdb_plot=True,      
        highlight_variants=False,
        save_plots=True,
        stops=False,
        threentdels=False,
        syn_zscore=False,
        domain_map=LMNA_domain_map,
        domain_colors=LMNA_domain_colors,
        start_pos=start_pos,
        end_pos=end_pos,
        wt_nuc_seq=LMNA_WT_nuc_seq,
        wt_aa_seq=LMNA_WT_aa_seq,
        remove_nonclassified_variants=False,
        input_pdb='../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/6jlb.pdb',
        pymol_view=lmna_pymol_view,
        image_size=(5000,2500),
        dpi=1000,
        pymol_exec="$HOME/pymol/pymol -c",
        extra_pymol="show sticks\n",
        dis_path='./consensus_plots_postreview/distribution/',
        heatmap_path='./consensus_plots_postreview/heatmap/',
        umap_path='./consensus_plots_postreview/umap/',
        pdb_path='./consensus_plots_postreview/pdb/',
        heatmap_figure_size=(40,20),
        vmin=0.6,
        vcenter=0.75,
        vmax=0.9,
        heatmap_color_scheme='YlOrBr',
        show_plots=False)
#!{pymol_command}


In [18]:
# Ellipticity vs circularity plot
# Plot the three major features against each other for iPS and neurons
feature_list_toplot = \
    [
        'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1',
        'Mean_Nuclei_AreaShape_FormFactor', 
        'Mean_Nuclei_AreaShape_Eccentricity',
        'Mean_Nuclei_AreaShape_Solidity'
    ]

axis_lims_features = \
    {
        l:(-10,10) for l in feature_list_toplot
    }

feature_rename = \
    {
        'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1':'Boundary Intensity',
        'Mean_Nuclei_AreaShape_FormFactor':'Circularity', 
        'Mean_Nuclei_AreaShape_Eccentricity':'Eccentricity',
        'Mean_Nuclei_AreaShape_Solidity':'Solidity'
    }

feature_pairs_toplot = list(itertools.combinations(feature_list_toplot, 2))
fig,axs = plt.subplots(figsize=(4*len(feature_pairs_toplot)+3,4),ncols=len(feature_pairs_toplot))
for i,pair in enumerate(feature_pairs_toplot):
    sns.scatterplot(
        data=variant_medians_merged.sort_values(by='Variant_Class'),
        x=pair[0],
        y=pair[1],
        hue='Variant_Class',
        palette=variant_type_palette,
        ax=axs[i],
        legend=(i==(len(feature_pairs_toplot)-1))
    )
    axs[i].set_xlabel(feature_rename[pair[0]], fontsize=24)
    axs[i].set_ylabel(feature_rename[pair[1]], fontsize=24)
    axs[i].set_xlim(axis_lims_features[pair[0]])
    axs[i].set_ylim(axis_lims_features[pair[1]])
plt.legend(loc='upper left',
           title='Variant Type',
           markerscale=2,
           bbox_to_anchor=(0.6,1),
           fontsize=14,
           title_fontsize=16)
plt.tight_layout()
fig.savefig('./consensus_plots_postreview/LMNAT3.pairwisenuclearshapeplot.022725.pdf',
            bbox_inches='tight')
plt.close()


In [19]:
# Plot by domain
variants_assayed_umap_toplot = \
    df_profiles_merged[df_profiles_merged['Variant_Class'].isin(['Missense','Synonymous'])]

# Annotate with domain information
variants_assayed_umap_toplot['position'] = \
    variants_assayed_umap_toplot['Variant'].map(lambda x: int(x[1:-1]))
variants_assayed_umap_toplot['Domain'] = \
    variants_assayed_umap_toplot\
        .apply(lambda row: 'Synon' if row['Variant_Class'] == 'Synonymous'\
                                else LMNA_domain_map[row['position']],
               axis=1)
fig,ax=plt.subplots(figsize=(4,4))
domain_cp = {**LMNA_domain_colors, 'Synon': 'darkgreen'}
sns.scatterplot(data=variants_assayed_umap_toplot, 
                x='UMAP1', y='UMAP2', 
                hue='Domain', 
                palette=domain_cp,
                s=25,
                ax=ax
               )

plt.legend(markerscale=2, fontsize=10, title='Domain', title_fontsize=12)
# Remove x-axis tick labels and ticks
plt.xticks([])
plt.xlabel('UMAP1', fontsize=16)

# Remove y-axis tick labels and ticks
plt.yticks([])
plt.ylabel('UMAP2', fontsize=16)
plt.tight_layout()

plt.close()
fig.savefig('./consensus_plots_postreview/LMNAT3.UMAP.domain.011325.pdf', 
            dpi=600)


In [20]:
# Look at variants based on where they are in structure
df_toplot = df_profiles_merged.query('Variant_Class in ["Synonymous","Missense"]')
df_toplot['Position'] = df_toplot['Variant'].map(lambda x: int(x[1:-1]))
df_toplot['Domain'] = \
    df_toplot\
        .apply(lambda row: 'Synon' if row['Variant_Class'] == 'Synonymous'\
                                else LMNA_domain_map[row['Position']],
               axis=1)

fig,ax=plt.subplots(figsize=(4,4))
domain_cp = {**LMNA_domain_colors, 'Synon': 'darkgreen'}
boxplot_with_significance(
    df=df_toplot,
    x_col='Domain',
    y_col='Morphological Impact Score',
    hue_col='Domain',
    palette=domain_cp,
    order=['Synon','Coil 1B','Linker 12','Coil 2A'],
    alpha=0.001,    # Show significance if p < 0.001
    pairs=list(itertools.combinations(['Synon','Coil 1B','Linker 12','Coil 2A'],2)),
    offset=0.02,
    ax=ax,
    show_points=True
)
plt.ylabel('Impact Score', fontsize=14)
plt.xlabel('', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.ylim(0,1)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/morphologicalimpact.bydomain.boxplot.112925.pdf')


In [21]:
# Look at variants based on where they are in structure
df_toplot = df_profiles_merged.query('Variant_Class in ["Synonymous","Missense"]')
df_toplot['Position'] = df_toplot['Variant'].map(lambda x: int(x[1:-1]))
df_toplot['Domain'] = \
    df_toplot\
        .apply(lambda row: 'Synon' if row['Variant_Class'] == 'Synonymous'\
                                else LMNA_domain_map[row['Position']],
               axis=1)
df_toplot = \
    df_toplot.merge(roc_auc_lmna, on='Variant')

fig,ax=plt.subplots(figsize=(4,4))
domain_cp = {**LMNA_domain_colors, 'Synon': 'darkgreen'}
boxplot_with_significance(
    df=df_toplot,
    x_col='Domain',
    y_col='AUC ROC',
    hue_col='Domain',
    palette=domain_cp,
    order=['Synon','Coil 1B','Linker 12','Coil 2A'],
    alpha=0.001,    # Show significance if p < 0.001
    pairs=list(itertools.combinations(['Synon','Coil 1B','Linker 12','Coil 2A'],2)),
    offset=0.02,
    ax=ax,
    show_points=True
)
plt.ylabel('Distinguishability Score', fontsize=14)
plt.xlabel('', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.ylim(0.45,1)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/auc_roc.bydomain.boxplot.112925.pdf')


In [22]:
# Combine Coil 1B + Coil 2A into "Coil", compare vs "Linker 12" via Fisher Exact Test
df_tmp = df_toplot.copy()

df_tmp["Domain2"] = df_tmp["Domain"].replace({
    "Coil 1B": "Coil",
    "Coil 2A": "Coil",
})

counts_domain = (df_tmp
    .query('Domain2 in ["Coil","Linker 12"]')
    .groupby(["Domain2", "Cluster"])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(2.5,3))
ax, frac, p, q, orr = plot_heatmap_fractional_bycluster(
    counts_domain,
    pseudocount=0,
    alpha=0.05,
    vmax=0.5,
    cmap="viridis",
    ax=ax,
    xlabel="",
    ylabel="Louvain cluster",
    normalize="within_group",
    cbar_label="Fraction variants in cluster"
)

# recolor xtick labels (update your palette dict accordingly)
label_to_palette_dict = {**domain_cp, "Coil": "black"}
for lab in ax.get_xticklabels():
    lab.set_color(label_to_palette_dict.get(lab.get_text(), "black"))

fig.savefig(
    "./consensus_plots_postreview/LMNAT3.coil_vs_linker12.louvainclusters.fraction_heatmap.022825.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.close()

In [ ]:
# Get tetramer and dimer interacting positions
neighbor_from_minima_dist=1.5 #distance from minimum to count as

#A11 structure Ahn et al 2019 first
nearest_dist_a11tetramer = \
    get_nearest_distance_dataframe('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/6jlb.pdb')
nearest_dist_a11tetramer['Ref_position'] = nearest_dist_a11tetramer['Ref_residue'].map(lambda x: int(x[1:]))
nearest_dist_a11tetramer['Nearest_position_chainA'] = nearest_dist_a11tetramer['Nearest_residue_chainA'].map(lambda x: int(x[1:]))
nearest_dist_a11tetramer['Nearest_position_chainB'] = nearest_dist_a11tetramer['Nearest_residue_chainB'].map(lambda x: int(x[1:]))
nearest_dist_a11tetramer['Nearest_position_chainD'] = nearest_dist_a11tetramer['Nearest_residue_chainD'].map(lambda x: int(x[1:]))
nearest_dist_a11tetramer_coil1B = \
    nearest_dist_a11tetramer\
        .query("Ref_position >= @start_pos & Ref_position <= @end_pos")\
        .query('Ref_position <= 222')

# Plot dimer positions for A11 - coil 1B
fig, axs = plt.subplots(nrows=3, figsize=(6,8))
axs[0].plot(nearest_dist_a11tetramer_coil1B['Ref_position'],
            nearest_dist_a11tetramer_coil1B['Dist_residue_chainD'],
            label="Distance",
            c='black')
axs[0].set_title('Intra-dimer C\u03b2 distance for A11 structure Coil 1B', fontsize=14)
axs[0].set_xlabel('Position',fontsize=12)
axs[0].set_ylabel('Distance',fontsize=12)

# Get the x and y data as numpy arrays
x = nearest_dist_a11tetramer_coil1B['Ref_position'].values
y = nearest_dist_a11tetramer_coil1B['Dist_residue_chainD'].values

# Identify local minima: indices where a point is lower than its immediate neighbors.
minima_indices = np.where((y[1:-1] < y[:-2]) & (y[1:-1] < y[2:]))[0] + 1

# Use a set to collect unique annotated positions.
annotated_positions = set()

# Mark and annotate local minima and their neighbors if the distance difference is less than 1.
for idx in minima_indices:
    # Mark and annotate the local minimum (red)
    annotated_positions.add(x[idx])
    axs[0].plot(x[idx], y[idx], 'bo')
    axs[0].annotate(f"{x[idx]}", (x[idx], y[idx]),
                textcoords="offset points", xytext=(0, -10), ha="center", color='#111D4A')
    
    # Check left neighbor, if it exists.
    if idx - 1 >= 0:
        diff_left = y[idx-1] - y[idx]
        if diff_left < neighbor_from_minima_dist:
            annotated_positions.add(x[idx+1])
            axs[0].plot(x[idx-1], y[idx-1], 'bo')  # Mark with blue circle.
            axs[0].annotate(f"{x[idx-1]}", (x[idx-1], y[idx-1]),
                        textcoords="offset points", xytext=(0, -10), ha="center", color='#111D4A')
    
    # Check right neighbor, if it exists.
    if idx + 1 < len(y):
        diff_right = y[idx+1] - y[idx]
        if diff_right < neighbor_from_minima_dist:
            annotated_positions.add(x[idx+1])
            axs[0].plot(x[idx+1], y[idx+1], 'bo')
            axs[0].annotate(f"{x[idx+1]}", (x[idx+1], y[idx+1]),
                        textcoords="offset points", xytext=(0, -10), ha="center", color='#111D4A')

coil1b_dimer_pos = list(sorted(annotated_positions))

# Plot tetramer positions for A11 - coil 1B
axs[1].plot(nearest_dist_a11tetramer_coil1B['Ref_position'],
            nearest_dist_a11tetramer_coil1B['Dist_residue_chainA'],
            label="Distance",
            c='black')
axs[1].set_title('Intra-multimer C\u03b2 distance for A11 structure Coil 1B',fontsize=14)
axs[1].set_xlabel('Position',fontsize=12)
axs[1].set_ylabel('Distance',fontsize=12)

# Get the x and y data as numpy arrays
x = nearest_dist_a11tetramer_coil1B['Ref_position'].values
y = nearest_dist_a11tetramer_coil1B['Dist_residue_chainA'].values

# Identify local minima: indices where a point is lower than its immediate neighbors.
minima_indices = np.where((y[1:-1] < y[:-2]) & (y[1:-1] < y[2:]))[0] + 1

# Use a set to collect unique annotated positions.
annotated_positions = set()

# Mark and annotate local minima and their neighbors if the distance difference is less than 1.
for idx in minima_indices:
    # Mark and annotate the local minimum (red)
    annotated_positions.add(x[idx])
    axs[1].plot(x[idx], y[idx], 'ro')
    axs[1].annotate(f"{x[idx]}", (x[idx], y[idx]),
                textcoords="offset points", xytext=(0, -10), ha="center", color='#92140C')
    
    # Check left neighbor, if it exists.
    if idx - 1 >= 0:
        diff_left = y[idx-1] - y[idx]
        if diff_left < neighbor_from_minima_dist:
            annotated_positions.add(x[idx+1])
            axs[1].plot(x[idx-1], y[idx-1], 'ro')  # Mark with blue circle.
            axs[1].annotate(f"{x[idx-1]}", (x[idx-1], y[idx-1]),
                        textcoords="offset points", xytext=(0, -10), ha="center", color='#92140C')
    
    # Check right neighbor, if it exists.
    if idx + 1 < len(y):
        diff_right = y[idx+1] - y[idx]
        if diff_right < neighbor_from_minima_dist:
            annotated_positions.add(x[idx+1])
            axs[1].plot(x[idx+1], y[idx+1], 'ro')
            axs[1].annotate(f"{x[idx+1]}", (x[idx+1], y[idx+1]),
                        textcoords="offset points", xytext=(0, -10), ha="center", color='#92140C')

coil1b_tetramer_pos = list(sorted(annotated_positions))

#Next, the multimer structure
nearest_dist_a22tetramer = \
    get_nearest_distance_dataframe("../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/molce-46-5-309-supple2.pdb")
nearest_dist_a22tetramer['Ref_position'] = nearest_dist_a22tetramer['Ref_residue'].map(lambda x: int(x[1:]))
nearest_dist_a22tetramer['Nearest_position_chainB'] = nearest_dist_a22tetramer['Nearest_residue_chainB'].map(lambda x: int(x[1:]))
nearest_dist_a22tetramer['Nearest_position_chainG'] = nearest_dist_a22tetramer['Nearest_residue_chainG'].map(lambda x: int(x[1:]))
nearest_dist_a22tetramer['Nearest_position_chainH'] = nearest_dist_a22tetramer['Nearest_residue_chainH'].map(lambda x: int(x[1:]))
nearest_dist_a22tetramer_coil2A = \
    nearest_dist_a22tetramer\
        .query("Ref_position >= @start_pos & Ref_position <= @end_pos")\
        .query('Ref_position >= 241')
nearest_dist_a22tetramer_coil2A['Dist_min_chainBGH'] = \
    nearest_dist_a22tetramer_coil2A.apply(lambda row: min(row['Dist_residue_chainB'], 
                                                          row['Dist_residue_chainG'], 
                                                          row['Dist_residue_chainH']), 
                                          axis=1)

# Plot tetramer positions for A22 - both A and H are important!
axs[2].plot(nearest_dist_a22tetramer_coil2A['Ref_position'],
            nearest_dist_a22tetramer_coil2A['Dist_residue_chainB'],
            label="Chain B",
            c='purple')
axs[2].plot(nearest_dist_a22tetramer_coil2A['Ref_position'],
            nearest_dist_a22tetramer_coil2A['Dist_residue_chainG'],
            label="Chain G",
            c='orange')
axs[2].plot(nearest_dist_a22tetramer_coil2A['Ref_position'],
            nearest_dist_a22tetramer_coil2A['Dist_residue_chainH'],
            label="Chain H",
            c='cyan')
axs[2].set_title('Intra-multimer C\u03b2 distance for A22 structure Coil 2A',fontsize=14)
axs[2].set_xlabel('Position',fontsize=12)
axs[2].set_ylabel('Distance',fontsize=12)

# Get the x and y data as numpy arrays
x = nearest_dist_a22tetramer_coil2A['Ref_position'].values
y = nearest_dist_a22tetramer_coil2A['Dist_min_chainBGH'].values

# Identify local minima: indices where a point is lower than its immediate neighbors.
minima_indices = np.where((y[1:-1] < y[:-2]) & (y[1:-1] < y[2:]))[0] + 1

# Use a set to collect unique annotated positions.
annotated_positions = set()

# Mark and annotate local minima and their neighbors if the distance difference is less than 1.
for idx in minima_indices:
    # Mark and annotate the local minimum (red)
    annotated_positions.add(x[idx])
    axs[2].plot(x[idx], y[idx], 'ro')
    axs[2].annotate(f"{x[idx]}", (x[idx], y[idx]),
                textcoords="offset points", xytext=(0, -10), ha="center", color='#92140C')
    
    # Check left neighbor, if it exists.
    if idx - 1 >= 0:
        diff_left = y[idx-1] - y[idx]
        if diff_left < neighbor_from_minima_dist:
            annotated_positions.add(x[idx+1])
            axs[2].plot(x[idx-1], y[idx-1], 'ro')  # Mark with blue circle.
            axs[2].annotate(f"{x[idx-1]}", (x[idx-1], y[idx-1]),
                        textcoords="offset points", xytext=(0, -10), ha="center", color='#92140C')
    
    # Check right neighbor, if it exists.
    if idx + 1 < len(y):
        diff_right = y[idx+1] - y[idx]
        if diff_right < neighbor_from_minima_dist:
            annotated_positions.add(x[idx+1])
            axs[2].plot(x[idx+1], y[idx+1], 'ro')
            axs[2].annotate(f"{x[idx+1]}", (x[idx+1], y[idx+1]),
                        textcoords="offset points", xytext=(0, -10), ha="center", color='#92140C')
plt.legend()
plt.ylim(4,20)
plt.tight_layout()
fig.savefig('./consensus_plots_postreview/LMNAT3.betacarbondistances.030225.pdf', dpi=600)
plt.close()
coil2a_tetramer_pos = list(sorted(annotated_positions))


In [25]:
# Tetramer and dimer interacting positions
palette_interacting = \
    {
        'Synon':'darkgreen',
        'Dimer-facing':'#111D4A',
        'Multimer-facing':'#92140C',
        'Both':'#52192B',
        'Neither':'lightgrey'
    }

variants_assayed_umap_toplot = \
    df_profiles_merged[df_profiles_merged['Variant_Class'].isin(['Missense','Synonymous'])]
variants_assayed_umap_toplot['Label'] = variants_assayed_umap_toplot\
    .apply(lambda row: 'Synon' if row['Variant_Class']=='Synonymous' \
                           else 'Both' if int(row['Variant'][1:-1]) in (set(coil1b_tetramer_pos).intersection(set(coil1b_dimer_pos))) \
                           else 'Multimer-facing' if int(row['Variant'][1:-1]) in (coil1b_tetramer_pos+coil2a_tetramer_pos) \
                           else 'Dimer-facing' if int(row['Variant'][1:-1]) in coil1b_dimer_pos \
                           else 'Neither', axis=1)
variants_assayed_umap_toplot['zorder'] = \
    variants_assayed_umap_toplot['Label'].map(lambda x: 0 if x=='Neither' else\
                                                  1 if x=='Synon' else 2)
variants_assayed_umap_toplot.sort_values('zorder',inplace=True)

# Look at subplots - faceted
fig,axs=plt.subplots(figsize=(6,3.3), ncols=2, nrows=1, sharex=False, sharey=False)
umap_pos_toplot = \
    variants_assayed_umap_toplot.copy()
pos_to_show = ['Dimer-facing']
palette_interacting_toshow = \
    {k: (palette_interacting[k] if k in pos_to_show else 'lightgrey') for k in palette_interacting.keys()}
umap_pos_toplot['pos'] = umap_pos_toplot['Label'].map(lambda x: 1 if x in pos_to_show else 0)
sns.scatterplot(data=umap_pos_toplot.sort_values('pos'), 
                x='UMAP1', 
                y='UMAP2', 
                hue='Label', 
                palette=palette_interacting_toshow,
                s=20,
                legend=False,
                ax=axs[0]
               )
axs[0].set_xticks([])
axs[0].set_yticks([])
axs[0].set_xlabel("UMAP 1", fontsize=16)
axs[0].set_ylabel("UMAP 2", fontsize=16)
axs[0].set_title('Dimer-facing', fontsize=16)

umap_pos_toplot = \
    variants_assayed_umap_toplot.copy()
pos_to_show = ['Multimer-facing']
palette_interacting_toshow = \
    {k: (palette_interacting[k] if k in pos_to_show else 'lightgrey') for k in palette_interacting.keys()}
umap_pos_toplot['pos'] = umap_pos_toplot['Label'].map(lambda x: 1 if x in pos_to_show else 0)
sns.scatterplot(data=umap_pos_toplot.sort_values('pos'), 
                x='UMAP1', 
                y='UMAP2', 
                hue='Label', 
                palette=palette_interacting_toshow,
                s=20,
                legend=False,
                ax=axs[1])
axs[1].set_xticks([])
axs[1].set_yticks([])
axs[1].set_xlabel("UMAP 1", fontsize=16)
axs[1].set_ylabel("UMAP 2", fontsize=16)
axs[1].set_title('Multimer-facing', fontsize=16)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/UMAP.bycoilinteraction.022525.pdf', dpi=600)


In [26]:
# Look at variants based on where they are in structure
fig,ax=plt.subplots(figsize=(4,4))
boxplot_with_significance(
    df=variants_assayed_umap_toplot,
    x_col='Label',
    y_col='Morphological Impact Score',
    hue_col='Label',
    palette=palette_interacting,
    order=['Synon','Dimer-facing','Multimer-facing','Both','Neither'],
    alpha=0.001,    # Show significance if p < 0.01
    pairs=[("Synon",s) for s in ['Dimer-facing','Multimer-facing','Neither']]+\
          [("Dimer-facing","Multimer-facing"),("Multimer-facing","Neither")],
    shared_label='***',
    offset=0.03,
    ax=ax,
    show_points=True
)
ax.set_xlabel("")
ax.set_ylabel('Impact Score', fontsize=14)
ax.tick_params(labelsize=11)
#ax.set_xticklabels(["Synon","Dimer\nfacing","Multimer\nfacing","Both","Neither"])
ax.set_xticklabels([s.get_text().replace('-','\n') for s in ax.get_xticklabels()])
plt.ylim(0,1.0)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/morphologicalimpact.byinteraction.boxplot.112925.pdf', dpi=600)


In [27]:
# Look at variants based on where they are in structure
fig,ax=plt.subplots(figsize=(4,4))
boxplot_with_significance(
    df=variants_assayed_umap_toplot.merge(roc_auc_lmna, on='Variant', how='inner'),
    x_col='Label',
    y_col='AUC ROC',
    hue_col='Label',
    palette=palette_interacting,
    order=['Synon','Dimer-facing','Multimer-facing','Both','Neither'],
    alpha=0.001,    # Show significance if p < 0.01
    pairs=[("Synon",s) for s in ['Dimer-facing','Multimer-facing','Neither']]+\
          [("Dimer-facing","Multimer-facing"),("Multimer-facing","Neither")],
    shared_label='***',
    offset=0.015,
    ax=ax,
    show_points=True
)
ax.set_xlabel("")
ax.set_ylabel('Distinguishability Score', fontsize=14)
ax.tick_params(labelsize=11)
#ax.set_xticklabels(["Synon","Dimer\nfacing","Multimer\nfacing","Both","Neither"])
ax.set_xticklabels([s.get_text().replace('-','\n') for s in ax.get_xticklabels()])
plt.ylim(0.45,1.0)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/auc_roc.byinteraction.boxplot.112925.pdf', dpi=600)


In [28]:
# Add interacting annotations
colorpalette_interacting = \
    {
        'Dimer-facing':'#111D4A',
        'Multimer-facing':'#92140C',
        'Both':'#52192B'
    }

# Combine positions from the three lists.
# Assume coil1b_dimer_pos, coil1b_tetramer_pos, and coil2a_tetramer_pos
# are lists of positions (integers).
all_interacting = set(coil1b_dimer_pos) | set(coil1b_tetramer_pos) | set(coil2a_tetramer_pos)

# Create the annotation dictionary.
# For each interacting position, label it 'D' if only in the dimer list,
# 'T' if only in one of the tetramer lists, and 'D+T' if in both.
interacting_annotation_dict = {}
for pos in all_interacting:
    in_dimer = pos in coil1b_dimer_pos
    in_tetramer = (pos in coil1b_tetramer_pos) or (pos in coil2a_tetramer_pos)
    if in_dimer and in_tetramer:
        interacting_annotation_dict[pos] = 'Both'
    elif in_dimer:
        interacting_annotation_dict[pos] = 'Dimer-facing'
    elif in_tetramer:
        interacting_annotation_dict[pos] = 'Multimer-facing'

# Now, for positions within the specified range that are not interacting,
# assign the label 'Other'.
for i in range(start_pos, end_pos + 1):
    if i not in interacting_annotation_dict:
        interacting_annotation_dict[i] = 'Neither'


In [ ]:
# Correlation between proline profiles only
df_variant_medianplusEMD_features_pca = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_averaged_medianplusEMD_PCA_021325.csv')
df_variant_medianplusEMD_features_pca.drop(columns='Unnamed: 0',inplace=True)
df_variant_medianplusEMD_features_pca['Variant_Class'] = \
    pd.Categorical(df_variant_medianplusEMD_features_pca['Variant'].astype(str).apply(variant_classification),
                    categories=mutation_types, ordered=True)
proline_PCA_embeddings = \
    df_variant_medianplusEMD_features_pca\
        .query('Variant_Class == "Missense"')
proline_PCA_embeddings['position'] = \
    proline_PCA_embeddings['Variant'].map(lambda s: int(s[1:-1]))
proline_PCA_embeddings['aaTo'] = \
    proline_PCA_embeddings['Variant'].map(lambda s: s[-1])
proline_PCA_embeddings = \
    proline_PCA_embeddings\
        .query('aaTo == "P"')\
        .drop(columns=['aaTo','Variant_Class','Variant'])

# Annotate with domain information
proline_PCA_embeddings['Domain'] = \
    proline_PCA_embeddings['position']\
        .map(lambda x: 'Synon' if x == 'Synonymous'\
                            else LMNA_domain_map[x])

# Proline palette
colorpalette_prolinecluster = ["#8E6E53","#A2AD59","#08605F"] #Fix manually based on the clustering, as it tends to randomly reorder the clusters
names_prolinecluster = {2:"\u03B1-helical",1:'Linker',0:'Linker-prox'} #Fix manually based on the clustering, as it tends to randomly reorder the clusters

# Annotate with Tetramer vs Dimer information
proline_PCA_embeddings['Interaction'] = \
    proline_PCA_embeddings['position']\
        .map(lambda x: 'Synon' if x == 'Synon'\
                            else interacting_annotation_dict[int(x)],
            )
colorpalette_interacting['Synon']='darkgreen'
colorpalette_interacting['Neither']='white'

# Make similarity heatmap
proline_PCA_embeddings, similarity_df, linkage = \
    plot_correlation_positions(
        proline_PCA_embeddings,
        metric='correlation',
        annotation_cols=['Domain','Interaction'],
        annotation_palettes=[LMNA_domain_colors,colorpalette_interacting],
        cluster_palette=colorpalette_prolinecluster,
        perform_knn_imputation=True,
        annotate_clusters=True,
        n_clust_pos=3,
        y_pos_fontsize=6.5,
        cbar_pos=(0.12, 0.5, 0.015, 0.3),
        plot_filename = "./consensus_plots_postreview/LMNAT3.PCAcorrelation.prolinesonly.aapos.V2.pdf",
        figsize=(10,10),
        cluster_spacing=0.03,
        cluster_names=names_prolinecluster,
        cluster_text_color={0:'white',1:'white',2:'white'},
        cluster_text_fontsize=14
    )


In [31]:
# Plot prolines on UMAP
variants_assayed_umap_toplot = \
    df_profiles_merged\
        .query('Variant_Class == "Missense"')

# Annotate with proline cluster
variants_assayed_umap_toplot['position'] = \
    variants_assayed_umap_toplot['Variant'].map(lambda x: int(x[1:-1]))
variants_assayed_umap_toplot['aato'] = \
    variants_assayed_umap_toplot['Variant'].map(lambda x: x[-1])
variants_assayed_umap_toplot_proline = \
    variants_assayed_umap_toplot\
        .query('aato == "P"')
variants_assayed_umap_toplot_proline = \
    variants_assayed_umap_toplot_proline\
        .merge(proline_PCA_embeddings.rename(columns={'Cluster':'Proline Cluster'}), on='position')

# UMAP
fig,ax=plt.subplots(figsize=(3,3))
sns.scatterplot(data=df_profiles_merged, 
                x='UMAP1', y='UMAP2', 
                c='lightgrey',
                s=25,
                ax=ax,
                legend=False
               )

sns.scatterplot(data=variants_assayed_umap_toplot_proline, 
                x='UMAP1', y='UMAP2', 
                hue='Proline Cluster', 
                palette=colorpalette_prolinecluster,
                s=300,
                marker='^',
                ax=ax,
                legend=False
               )

# Remove x-axis tick labels and ticks
plt.xticks([])
plt.xlabel('UMAP 1', fontsize=16)

# Remove y-axis tick labels and ticks
plt.yticks([])
plt.ylabel('UMAP 2', fontsize=16)

# Add title
#plt.title('Proline Substitutions\nby Domain on UMAP', fontsize=16)f

plt.tight_layout()

plt.close()
fig.savefig('./consensus_plots_postreview/LMNAT3.UMAP.prolinecluster.022025.pdf',dpi=600)


In [32]:
# Same heatmap for proline clusters
variant_profiles_toplot = \
    df_profiles_merged.copy()
variant_profiles_toplot = \
    variant_profiles_toplot\
        .query('Variant_Class == "Missense"')
variant_profiles_toplot['position'] = \
    variant_profiles_toplot['Variant'].map(lambda x: int(x[1:-1]))
variant_profiles_toplot = \
    variant_profiles_toplot\
        .query('position in @proline_PCA_embeddings["position"].values')\
        .merge(proline_PCA_embeddings[['position','Cluster']]\
                   .rename(columns={'Cluster':'Position Cluster'}),
                   on='position')
cluster_labels = ['Linker-\nprox.','Linker','\u03B1-\nhelical']
variant_profiles_toplot['Position Cluster'] = \
    pd.Categorical(
        variant_profiles_toplot['Position Cluster'].map(lambda x: cluster_labels[x]),
        categories=['Linker','Linker-\nprox.','\u03B1-\nhelical'],
        ordered=True
    )
variant_profiles_toplot.sort_values(by='Position Cluster', inplace=True)
df_proline = variant_profiles_toplot.copy()

counts_proline = variant_profiles_toplot\
            .groupby(['Position Cluster','Cluster'])\
            .size()\
            .unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(3,3))
ax, frac, p, q, orr = plot_heatmap_fractional_bycluster(
    counts_proline,          # your existing proline cluster by Louvain cluster table
    pseudocount=0,
    alpha=0.05,
    vmax=0.5,
    cmap="viridis",
    ax=ax,
    xlabel="",
    ylabel="Louvain cluster",
    normalize="within_group",
    cbar_label="Fraction variants in cluster"
)
label_to_palette_dict   = {cluster_labels[i]:colorpalette_prolinecluster[i] for i in range(len(cluster_labels))}
xticklabels = ax.get_xticklabels()
for label, color in zip(xticklabels, [label_to_palette_dict[lab.get_text()] for lab in xticklabels]):
    label.set_color(color)

fig.savefig(
    "./consensus_plots_postreview/LMNAT3.prolinecluster.louvainclusters.fraction_heatmap.022825.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.close()

In [ ]:
# Now, cluster only those positions in the coil cluster by average profiles
coilpos_tocluster_byprolinePCA = \
    proline_PCA_embeddings.query('Cluster == 2')['position'].values

# Plot cluster plot over positions, averaged variants - PCA space and correlation distance
# Subset Missense
colorpalette_positioncluster = \
    [ #Fix manually based on the clustering, as it tends to randomly reorder the clusters
        "#6B62CB", #change this color scheme
        "#E91CB2", #change this color scheme
        "#91BAD6"  #change this color scheme
    ]
df_variant_medianplusEMD_features_pca = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_averaged_medianplusEMD_PCA_021325.csv')
df_variant_medianplusEMD_features_pca.drop(columns='Unnamed: 0', inplace=True)
df_variant_medianplusEMD_features_pca['Variant_Class'] = \
    pd.Categorical(df_variant_medianplusEMD_features_pca['Variant'].astype(str).apply(variant_classification),
                    categories=mutation_types, ordered=True)
n_comp_pca=12
n_variants_perpos_filter=15

variants_assayed_umap_toplot_missenseonly = \
    df_variant_medianplusEMD_features_pca\
        .query("Variant_Class == 'Missense'")
variants_assayed_umap_toplot_missenseonly['position'] = \
    variants_assayed_umap_toplot_missenseonly['Variant'].map(lambda s: int(s[1:-1]))
aapos_represented = \
    (variants_assayed_umap_toplot_missenseonly['position'].value_counts() >= n_variants_perpos_filter)
aapos_represented = aapos_represented[aapos_represented].index
aapos_represented = sorted(aapos_represented)
variants_assayed_umap_toplot_missenseonly = \
    variants_assayed_umap_toplot_missenseonly\
        .query('position.isin(@aapos_represented)')\
        .query('position.isin(@coilpos_tocluster_byprolinePCA)')

# Perform profile averaging
pca_dim = ['PC'+str(i) for i in range(1,n_comp_pca+1)]
syn_profile = \
    df_variant_medianplusEMD_features_pca[pca_dim+['Variant_Class']]\
        .query('Variant_Class == "Synonymous"')\
        .drop(columns='Variant_Class')\
        .mean()
syn_profile['position']='Synon'
pos_averaged_profiles = \
    variants_assayed_umap_toplot_missenseonly[pca_dim+['position']]\
        .groupby('position')\
        .mean()\
        .reset_index()
pos_averaged_profiles['position'] = pos_averaged_profiles['position'].astype(str)
pos_averaged_profiles.loc[-1] = syn_profile.T

# Annotate with domain information
pos_averaged_profiles['Domain'] = \
    pos_averaged_profiles['position']\
        .map(lambda x: 'Synon' if x == 'Synon'\
                            else LMNA_domain_map[int(x)],
            )

# Annotate with Tetramer vs Dimer information
pos_averaged_profiles['Interaction'] = \
    pos_averaged_profiles['position']\
        .map(lambda x: 'Synon' if x == 'Synon'\
                            else interacting_annotation_dict[int(x)],
            )
colorpalette_interacting['Synon']='darkgreen'
domain_cp = {**LMNA_domain_colors, 'Synon': 'darkgreen'}
names_coilcluster = {0:'Low Impact',1:'Agg.-sensitive',2:'Abund.-sensitive'}

# Make correlation heatmap
nclust_coilpos=3
coilpos_averaged_profiles, similarity_df, linkage = \
    plot_correlation_positions(
        pos_averaged_profiles,
        metric='correlation',
        annotation_cols=['Domain','Interaction'],
        annotation_palettes=[domain_cp,colorpalette_interacting],
        cluster_palette=colorpalette_positioncluster,
        perform_knn_imputation=False,
        fillnaval=0,
        annotate_clusters=True,
        n_clust_pos=nclust_coilpos,
        y_pos_fontsize=8,
        cbar_pos=(0.12, 0.65, 0.015, 0.15),
        plot_filename = "./consensus_plots_postreview/LMNAT3.PCAcorrelation.coilpositionsonly.averagedprofiles.aapos.V2.032325.pdf",
        figsize=(8,8),
        cluster_spacing=0.05,
        cluster_names=names_coilcluster,
        cluster_text_color={0:'white',1:'white',2:'white'},
        cluster_text_fontsize=11
    )


In [ ]:
# Plot clusters on structure - A11 and A22
input_to_paint = \
    pos_averaged_profiles\
        [['position','Cluster']]\
        .reset_index(drop=True)\
        .query('position != "Synon"')\
        .rename(columns={'Cluster':'score'})
input_to_paint['position'] = input_to_paint['position'].astype(int)
input_to_paint_toplot = input_to_paint.copy()
input_to_paint_toplot['score'] = input_to_paint_toplot['score'] + 1

# Paint all colors at once
lmna_pymol_view = \
    '''set_view (\
        -0.915147185,    0.403044611,   -0.007183676,\
        -0.175614133,   -0.414666086,   -0.892865419,\
        -0.362844229,   -0.815845907,    0.450264663,\
         0.000000000,    0.000000000, -382.015167236,\
      -190.245574951, -108.335136414,  109.851905823,\
       301.183654785,  462.846679688,  -20.000000000 )'''
start_pos=178
end_pos=273
n_clust_pos=3
pymol_command = \
    paint_structure(
        input_to_paint_toplot,
        output_suffix='LMNA_dimer_PCAaveragedclusters',
        output_dir="./consensus_plots_postreview/pdb/",
        input_pdb='../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/6jlb.pdb',
        pymol_view=lmna_pymol_view,
        start_pos=start_pos,
        end_pos=end_pos,
        pymol_exec="$HOME/pymol/pymol -c",
        image_size=(5000,2500),
        dpi=1000,
        vmin=0,
        vmax=3,
        extra_pymol="show spheres\n",
        discrete_coloring=True,
        discrete_colors=["#2F2F2F"]+colorpalette_positioncluster
        )
#!{pymol_command} #uncomment me to run pymol!

# Paint all colors at once
lmna_pymol_view2 = \
    '''set_view (\
         0.128099561,   -0.553015828,    0.823264420,\
         0.121202230,    0.832608640,    0.540436924,\
        -0.984324038,    0.030553162,    0.173684672,\
         0.000030637,   -0.000129715, -390.408966064,\
      -195.511444092,  -99.926048279, -157.834701538,\
        82.949302673,  697.868286133,  -20.000000000 )'''
start_pos=178
end_pos=273
n_clust_pos=3
pymol_command = \
    paint_structure(
        input_to_paint_toplot,
        output_suffix='LMNA_A22_PCAaveragedclusters',
        output_dir="./consensus_plots_postreview/pdb/",
        input_pdb='../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/molce-46-5-309-supple2.pdb',
        pymol_view=lmna_pymol_view2,
        start_pos=start_pos,
        end_pos=end_pos,
        pymol_exec="$HOME/pymol/pymol -c",
        image_size=(5000,2500),
        dpi=1000,
        extra_pymol="show sticks\n",
        vmin=0,
        vmax=n_clust_pos,
        discrete_coloring=True,
        discrete_colors=["#2F2F2F"]+colorpalette_positioncluster
)
#!{pymol_command} #uncomment me to run pymol!


In [ ]:
# Plot domain stacked bar chart of clusters
# Create stacked bar plot
# Step 1: Get the counts per Domain per Cluster
pos_average_tocount = \
    coilpos_averaged_profiles[['Domain','Cluster']]\
        .query('Domain != "Synon"')
pos_average_tocount['Domain'] = \
    pd.Categorical(pos_average_tocount['Domain'],categories=['Coil 1B','Coil 2A'],ordered=True)
counts = pos_average_tocount\
            .groupby(['Domain', 'Cluster'])\
            .size()\
            .unstack(fill_value=0)

# Step 3: Define your color palette for clusters
colorpalette_positioncluster = { #Fix manually based on the clustering, as it tends to randomly reorder the clusters
    0: "#6B62CB",
    1: "#E91CB2",
    2: "#91BAD6"
}
pairs = [('Coil 1B','Coil 2A')]
fig,ax=plt.subplots(figsize=(2.5,2.5))
ax = stacked_bar_with_proportion_significance(
    counts, 
    pairs, 
    alpha=0.01,
    shared_label="**",
    title='',
    ylabel='# Positions',
    xlabel='',
    ax=ax,
    offset=0.1,
    colorpalette=colorpalette_positioncluster,
    pseudocount=1
)
plt.tight_layout()
plt.ylabel('# Positions',fontsize=16)
plt.yticks(fontsize=14)
fig.savefig('./consensus_plots_postreview/LMNAT3.deepcoilposition.clustersbydomain.022825.pdf', dpi=600)
plt.close()


In [ ]:
# Plot domain stacked bar chart of clusters
# Create stacked bar plot
# Step 1: Get the counts per Domain per Cluster
pos_average_tocount = \
    coilpos_averaged_profiles[['Interaction','Cluster']]\
        .query('Interaction != "Synon"')
pos_average_tocount['Interaction'] = \
    pd.Categorical(pos_average_tocount['Interaction'],categories=['Dimer-facing','Multimer-facing','Both','Neither'],ordered=True)
counts = pos_average_tocount\
            .groupby(['Interaction', 'Cluster'])\
            .size()\
            .unstack(fill_value=0)

# Step 2: Define your color palette for clusters
colorpalette_positioncluster = { #Fix manually based on the clustering, as it tends to randomly reorder the clusters
    0: "#6B62CB",
    1: "#E91CB2",
    2: "#91BAD6"
}

# Step 3: Create the stacked bar plot
fig, ax = plt.subplots(figsize=(3.5,2.5))
pairs = [('Multimer-facing','Dimer-facing'),('Multimer-facing','Neither'),('Dimer-facing','Neither')]
ax = stacked_bar_with_proportion_significance(
    counts, 
    pairs, 
    alpha=0.01,
    shared_label="**",
    title='',
    ylabel='# Positions',
    xlabel='',
    ax=ax,
    offset=0.05,
    colorpalette=colorpalette_positioncluster,
    pseudocount=1
)
ax.set_xticklabels(["Dimer\nfacing","Multimer\nfacing","Both","Neither"])
plt.ylabel('# Positions',fontsize=16)
plt.yticks(fontsize=14)
plt.tight_layout()
fig.savefig('./consensus_plots_postreview/LMNAT3.deepcoilposition.clustersbyinteraction.022825.pdf', dpi=600)
plt.close()


In [ ]:
#### Plot clusters of coil positions on UMAP - density plot
variant_profiles_toplot = \
    df_profiles_merged.copy()
variant_profiles_singlemissense_toplot = \
    variant_profiles_toplot\
        .query('Variant_Class == "Missense"')
variant_profiles_singlemissense_toplot['position'] = \
    variant_profiles_singlemissense_toplot['Variant'].map(lambda x: (x[1:-1]))
cluster_labels = ['Low Impact','Aggregation-sensitive','Abundance-sensitive']
clusters_in_order = ['Aggregation-sensitive','Abundance-sensitive','Low Impact']
clusters_in_order_num = [1,2,0] #Fix manually based on the clustering, as it tends to randomly reorder the clusters

# Plot clusters on UMAP separately
fig,axs=plt.subplots(figsize=(3,9),nrows=3,sharex=False,sharey=False)
for k,c in enumerate(clusters_in_order_num):
    pos_in_cluster = \
        coilpos_averaged_profiles[['position','Cluster']]\
            .query('position != "Synonymous"')\
            .query('Cluster == @c')\
            ['position'].values
    ax=axs[k]
    sns.scatterplot(data=variant_profiles_toplot, 
                    x='UMAP1', 
                    y='UMAP2', 
                    c='lightgrey',
                    s=20,
                    ax=ax,
                    legend=False)
    sns.kdeplot(data=variant_profiles_singlemissense_toplot.query('position in @pos_in_cluster'), 
                x="UMAP1", 
                y="UMAP2", 
                color=colorpalette_positioncluster[c], 
                fill=False,
                ax=ax,
                legend=False
               )
    ax.set_ylabel('UMAP 2',fontsize=16)
    ax.set_xlabel('UMAP 1',fontsize=16)
    ax.set_title(clusters_in_order[k], fontsize=16)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
fig.savefig('./consensus_plots_postreview/LMNAT3.UMAP.coilpositionclusters.density.022025.pdf', dpi=600)
plt.close()


In [63]:
# Look at distribution of mutations at deep coil clusters on Louvain Clusters
n_louvain_clust=10
variant_profiles_toplot = \
    df_profiles_merged.copy()
variant_profiles_toplot = \
    variant_profiles_toplot\
        .query('Variant_Class == "Missense"')
variant_profiles_toplot['position'] = \
    variant_profiles_toplot['Variant'].map(lambda x: (x[1:-1]))
variant_profiles_toplot = \
    variant_profiles_toplot\
        .query('position in @coilpos_averaged_profiles["position"].values')\
        .merge(coilpos_averaged_profiles[['position','Cluster']]\
                   .rename(columns={'Cluster':'Position Cluster'}),
                   on='position')
cluster_labels = ['Low\nImpact','Agg.-\nSens.','Abund.-\nSens.']
variant_profiles_toplot['Position Cluster'] = \
    pd.Categorical(
        variant_profiles_toplot['Position Cluster'].map(lambda x: cluster_labels[x]),
        categories=['Agg.-\nSens.','Abund.-\nSens.','Low\nImpact'],
        ordered=True
    )
variant_profiles_toplot.sort_values(by='Position Cluster', inplace=True)
df_coil = variant_profiles_toplot.copy()
counts = variant_profiles_toplot\
            .groupby(['Position Cluster','Cluster'])\
            .size()\
            .unstack(fill_value=0)

# Create the stacked bar plot
fig, ax = plt.subplots(figsize=(4,3.5))
pairs = [('Low\nImpact','Abund.-\nSens.'),('Agg.-\nSens.','Abund.-\nSens.'),('Low\nImpact','Agg.-\nSens.')]
ax = stacked_bar_with_proportion_significance(
    counts, 
    pairs, 
    alpha=0.001,
    shared_label="***",
    title='',
    ylabel='# Variants',
    xlabel='',
    ax=ax,
    offset=0.05,
    colorpalette={(i+1):sns.color_palette()[i] for i in range(n_louvain_clust)},
    pseudocount=1
)
plt.legend(
    bbox_to_anchor=(1,1.05),
    handles=[Line2D([0], [0], 
                    marker='o', 
                    color='w',
                    markerfacecolor=sns.color_palette()[i],
                    label=str(i+1), 
                    markersize=12) \
             for i in range(n_louvain_clust)],
    fontsize=11,
    title="Louvain\nCluster",
    title_fontsize=11
          )
plt.tight_layout()
plt.ylabel('# Variants',fontsize=16)
plt.yticks(fontsize=12)
fig.savefig('./consensus_plots_postreview/LMNAT3.deepcoilposition.louvainclusters.022825.pdf', dpi=600)
plt.close()

# Plot heatmap
fig, ax = plt.subplots(figsize=(3,3))
ax, frac, p, q, orr = plot_heatmap_fractional_bycluster(
    counts,          # your existing deep-coil × Louvain table
    pseudocount=0,
    alpha=0.05,
    vmax=0.5,
    cmap="viridis",
    ax=ax,
    xlabel="",
    ylabel="Louvain cluster",
    normalize="within_group",
    cbar_label="Fraction variants in cluster"
)
label_to_palette_dict   = {cluster_labels[i]:colorpalette_positioncluster[i] for i in range(len(cluster_labels))}
xticklabels = ax.get_xticklabels()
for label, color in zip(xticklabels, [label_to_palette_dict[lab.get_text()] for lab in xticklabels]):
    label.set_color(color)

fig.savefig(
    "./consensus_plots_postreview/LMNAT3.deepcoilposition.louvainclusters.fraction_heatmap.022825.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.close()

In [47]:
# Look at variants based on where they are in structure
fig,ax=plt.subplots(figsize=(3,4))
posclusters=['Agg.-\nSens.','Abund.-\nSens.','Low\nImpact']
ah_cluster_palette = \
    {'Low\nImpact':colorpalette_positioncluster[0],
     'Agg.-\nSens.':colorpalette_positioncluster[1],
     'Abund.-\nSens.':colorpalette_positioncluster[2]}
boxplot_with_significance(
    df=variant_profiles_toplot,
    x_col='Position Cluster',
    y_col='Morphological Impact Score',
    hue_col='Position Cluster',
    palette=ah_cluster_palette,
    order=posclusters,
    alpha=0.001,    # Show significance if p < 0.001,
    pairs=list(itertools.combinations(posclusters, 2)),
    shared_label='***',
    offset=0.03,
    ax=ax,
    show_points=True
)
ax.set_xlabel("")
ax.set_ylabel('Impact Score', fontsize=14)
ax.tick_params(labelsize=12)
#ax.set_title('Deep Coil Variant\n Impacts by Position', fontsize=14)
plt.ylim(0,1.0)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/morphologicalimpact.bydeepcoilpositioncluster.boxplot.112825.pdf', dpi=600)


In [48]:
# Look at variants based on where they are in structure
fig,ax=plt.subplots(figsize=(3,4))
boxplot_with_significance(
    df=variant_profiles_toplot.merge(roc_auc_lmna, on='Variant', how='inner'),
    x_col='Position Cluster',
    y_col='AUC ROC',
    hue_col='Position Cluster',
    palette=ah_cluster_palette,
    order=posclusters,
    alpha=0.001,    # Show significance if p < 0.001,
    pairs=list(itertools.combinations(posclusters, 2)),
    shared_label='***',
    offset=0.03,
    ax=ax,
    show_points=True
)
ax.set_xlabel("")
ax.set_ylabel('Distinguishability Score', fontsize=14)
ax.tick_params(labelsize=12)
#ax.set_title('Deep Coil Variant\n Impacts by Position', fontsize=14)
plt.ylim(0.45,1)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/auc_roc.bydeepcoilpositioncluster.boxplot.112925.pdf', dpi=600)


In [49]:
# Heatmaps: AA (rows) x position clusters (cols) for MIS, AUROC, landmark features
# Create df combining proline clusters and coil clusters
landmark_features = \
    [
        'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1',
        'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1',
        'Mean_NucleiExpanded_Granularity_1_CH1',
        'Mean_Nuclei_AreaShape_FormFactor',
    ]
aliphatic = ["A", "V", "I", "L", "G"]
aromatic = ["F", "Y", "W"]
nonpolar = ["C", "M", "P"]
polar = ["S", "T", "N", "Q"]
acidic = ["D", "E"]
basic = ["H", "K", "R"]
aa_order = aliphatic + aromatic + nonpolar + polar + acidic + basic

df_toplot = \
    df_proline[['Variant','Position Cluster','position','Morphological Impact Score']]\
        .rename(columns={'Position Cluster':'ProlineCluster'})\
        .merge(variant_medians_merged, on='Variant')\
        .merge(roc_auc_lmna, on='Variant')\
        .merge(df_coil[['Variant','Position Cluster']], on='Variant', how='left')\
        .rename(columns={'Position Cluster':'CoilCluster'})
proline_cluster_order   = ['Linker', 'Linker-\nprox.', '\u03B1-\nhelical']
proline_cluster_labels  = ['Linker', 'Lprox', '\u03B1H']
proline_cluster_palette = {'Linker-\nprox.':colorpalette_prolinecluster[0], 'Linker':colorpalette_prolinecluster[1], '\u03B1-\nhelical':colorpalette_prolinecluster[2]}
coil_cluster_order      = ['Agg.-\nSens.', 'Abund.-\nSens.', 'Low\nImpact']
coil_cluster_labels     = ['Agg', 'Abund', 'LI']
coil_cluster_palette    = ah_cluster_palette
label_to_palette_dict   = {coil_cluster_labels[i]:coil_cluster_palette[coil_cluster_order[i]] for i in range(len(coil_cluster_labels))} |\
                              {proline_cluster_labels[i]:proline_cluster_palette[proline_cluster_order[i]] for i in range(len(proline_cluster_labels))}
df_posanno = df_toplot.copy()
df_posanno['ProlineCluster'] = df_posanno['ProlineCluster']\
    .map(lambda x: 'Linker' if x=='Linker' else 'Linker-proximal' if x=='Linker-\nprox.' else '\u03B1-helical' if x=='\u03B1-\nhelical' else np.nan)
df_posanno['CoilCluster'] = df_posanno['CoilCluster']\
    .map(lambda x: 'Aggregation-sensitive' if x=='Agg.-\nSens.' else 'Abundance-sensitive' if x=='Abund.-\nSens.' else 'Low Impact' if x=='Low\nImpact' else np.nan)

# Set aa_to
df_toplot['aa_to'] = df_toplot['Variant'].map(lambda x: x[-1])

# Features
features_for_heatmap = ['Morphological Impact Score', 'AUC ROC'] + landmark_features
feature_limits = {
    'Morphological Impact Score': (0.5, 0.8),
    'AUC ROC': (0.6, 0.9),
    'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1': (-5, 5),
    'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1': (-5, 5),
    'Mean_NucleiExpanded_Granularity_1_CH1': (-10, 10),
    'Mean_Nuclei_AreaShape_FormFactor': (-5, 5),
}
name_features_hm = {
    'Morphological Impact Score':'Impact\nScore',
    'AUC ROC':'Distinguishability\nScore',
    'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1':'Lamin A Nuc\nIntensity',
    'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1':'Lamin A Nuc\nBndry Intensity',
    'Mean_NucleiExpanded_Granularity_1_CH1':'Lamin A Nuc\nGranularity 1',
    'Mean_Nuclei_AreaShape_FormFactor':'Nuc Circularity'
}

# Make large figure
nfeat = len(features_for_heatmap)

fig = plt.figure(figsize=(9, 4))
gs = fig.add_gridspec(
    nrows=2, ncols=nfeat,
    height_ratios=[1.0, 0.10],   # 2nd row reserved for colorbars
    hspace=0.10,
    wspace=0.55                  # <-- key: more space between columns
)

heat_axes = [fig.add_subplot(gs[0, j]) for j in range(nfeat)]
cbar_cells = [fig.add_subplot(gs[1, j]) for j in range(nfeat)]

for j, feat in enumerate(features_for_heatmap):
    ax = heat_axes[j]
    ccell = cbar_cells[j]
    ccell.set_axis_off()  # we’ll draw the bar inside a centered inset

    # --- build pivot_with_spacer as you already do ---
    pivot_proline = (
        df_toplot.groupby(['aa_to', 'ProlineCluster'])[feat]
        .mean().unstack(fill_value=np.nan).reindex(index=aa_order)
    )
    pivot_coil = (
        df_toplot.groupby(['aa_to', 'CoilCluster'])[feat]
        .mean().unstack(fill_value=np.nan).reindex(index=aa_order)
    )
    pivot = pd.concat([pivot_proline, pivot_coil], axis=1)

    pivot_with_spacer = pivot[proline_cluster_order + coil_cluster_order].copy()
    pivot_with_spacer[' '] = np.nan
    col_order = proline_cluster_order + [' '] + coil_cluster_order
    pivot_with_spacer = pivot_with_spacer.reindex(columns=col_order)
    col_labels = proline_cluster_labels + [' '] + coil_cluster_labels

    # --- limits ---
    vmin, vmax = feature_limits.get(
        feat,
        (np.nanmin(pivot_with_spacer.values), np.nanmax(pivot_with_spacer.values))
    )
    mid = (vmin + vmax) / 2

    # --- heatmap ---
    feature_title = name_features_hm.get(feat, feat)
    hm = sns.heatmap(
        pivot_with_spacer,
        cmap="binary" if ("Score" in feature_title) else "bwr",
        vmin=vmin, vmax=vmax,
        linewidths=0.5, linecolor="white",
        square=True,
        cbar=False,
        ax=ax
    )

    # --- centered, shrunken colorbar inset inside its own grid cell ---
    # [left, bottom, width, height] in the colorbar-cell axes coordinates
    cax = ccell.inset_axes([0.1, 0.05, 0.8, 0.3])  # <-- true x-shrink (80% width)

    cbar = fig.colorbar(hm.collections[0], cax=cax, orientation="horizontal")
    cbar.set_ticks([vmin, mid, vmax])

    # Shorter labels fix the overlap (e.g., -10, 0, 10 instead of -10.0 ...)
    cbar.ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%g'))
    cbar.ax.tick_params(labelsize=10, pad=1)

    # --- x ticks (skip spacer) ---
    keep = [i for i, lbl in enumerate(col_labels) if lbl.strip()]
    ax.set_xticks([k + 0.5 for k in keep])
    xticklabels_text = [col_labels[i] for i in keep]
    ax.set_xticklabels(xticklabels_text, rotation=90, fontsize=7)
    xticklabels = ax.get_xticklabels()
    # Iterate and set color for each label
    for label, color in zip(xticklabels, [label_to_palette_dict[lab] for lab in xticklabels_text]):
        label.set_color(color)

    # --- y ticks & titles ---
    # --- y ticks: set locations to match the actual heatmap rows ---
    nrows = pivot_with_spacer.shape[0]
    ax.set_yticks(np.arange(nrows) + 0.5)
    
    # if you truly want aa_order regardless, ensure lengths match:
    ylabels = list(pivot_with_spacer.index)   # safest: labels that match rows
    ax.set_yticklabels(ylabels, rotation=0, fontsize=9)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(feature_title, fontsize=10)

# IMPORTANT: avoid bbox_inches='tight' re-squeezing everything around text
fig.savefig("./consensus_plots_postreview/LMNAT3.coilclusters_features.heatmap.fixed.pdf",
            dpi=150)
plt.close()


In [ ]:
# Remake heatmaps with annotations
proline_annotation_dict = \
    pd.Series(proline_PCA_embeddings['Cluster'].values,
              index=proline_PCA_embeddings['position'].values)\
        .to_dict()
for i in range(start_pos,end_pos+1):
    if i not in proline_annotation_dict.keys():
        proline_annotation_dict[i]=-1
colorpalette_prolinecluster_dict = {}
for i in range(len(colorpalette_prolinecluster)):
    colorpalette_prolinecluster_dict[i]=colorpalette_prolinecluster[i]
ahelical_annotation_dict = \
    pd.Series(coilpos_averaged_profiles.query('position != "Synon"')['Cluster'].values,
              index=coilpos_averaged_profiles.query('position != "Synon"')['position'].values.astype(int))\
        .to_dict()
for i in range(start_pos,end_pos+1):
    if i not in ahelical_annotation_dict.keys():
        ahelical_annotation_dict[i]=-1
colorpalette_prolinecluster_dict[-1] = 'white'
colorpalette_positioncluster[-1] = 'white'

# Make heatmaps and other plots
feature_to_heatmap = 'Mean_Nuclei_AreaShape_FormFactor'
feature_scores, pymol_command = \
    plot_feature(variant_medians_merged,
                 umap_coords,
                 feature_to_heatmap,
                 "LMNA Tile 3 Nuclear Shape Factor",
                 "LMNA.T3.NucleiShapeFactor",
                  dist_plot=True,
                  umap_plot=True,
                  heatmap_plot=True,
                  pdb_plot=True,      
                  highlight_variants=False,
                  save_plots=True,
                  stops=False,
                  threentdels=False,
                  syn_zscore=False,
                  domain_map=LMNA_domain_map,
                  domain_colors=LMNA_domain_colors,
                  anno_height=0.02,
                  other_anno_map=[proline_annotation_dict,ahelical_annotation_dict,interacting_annotation_dict],
                  other_anno_colors=[colorpalette_prolinecluster_dict,colorpalette_positioncluster,colorpalette_interacting],
                  start_pos=start_pos,
                  end_pos=end_pos,
                  wt_nuc_seq=LMNA_WT_nuc_seq,
                  wt_aa_seq=LMNA_WT_aa_seq,
                  remove_nonclassified_variants=False,
                  input_pdb='../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/6jlb.pdb',
                  pymol_view=lmna_pymol_view,
                  image_size=(5000,2500),
                  dpi=1000,
                  pymol_exec="$HOME/pymol/pymol -c",
                  extra_pymol="show sticks\n",
                  dis_path='./consensus_plots_postreview/distribution/',
                  heatmap_path='./consensus_plots_postreview/heatmap/',
                  umap_path='./consensus_plots_postreview/umap/',
                  pdb_path='./consensus_plots_postreview/pdb/',
                  heatmap_figure_size=(40,20),
                  vmin=-6,
                  vmax=6,
                  show_plots=False
                )


In [ ]:
# Look at specific features, adding domain annotation onto heatmaps
feature_to_heatmap = 'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1'
feature_scores, pymol_command = \
    plot_feature(variant_medians_merged,
                 umap_coords,
                 feature_to_heatmap,
                 "LMNA Tile 3 Nuclear Abundance",
                 "LMNA.T3.abundance",
                  dist_plot=True,
                  umap_plot=True,
                  heatmap_plot=True,
                  pdb_plot=True,      
                  highlight_variants=False,
                  save_plots=True,
                  stops=False,
                  threentdels=False,
                  syn_zscore=False,
                  domain_map=LMNA_domain_map,
                  domain_colors=LMNA_domain_colors,
                  anno_height=0.02,
                  other_anno_map=[proline_annotation_dict,ahelical_annotation_dict,interacting_annotation_dict],
                  other_anno_colors=[colorpalette_prolinecluster_dict,colorpalette_positioncluster,colorpalette_interacting],
                  start_pos=start_pos,
                  end_pos=end_pos,
                  wt_nuc_seq=LMNA_WT_nuc_seq,
                  wt_aa_seq=LMNA_WT_aa_seq,
                  remove_nonclassified_variants=False,
                  input_pdb='../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/6jlb.pdb',
                  pymol_view=lmna_pymol_view,
                  image_size=(5000,2500),
                  dpi=1000,
                  pymol_exec="$HOME/pymol/pymol -c",
                  extra_pymol="show sticks\n",
                  dis_path='./consensus_plots_postreview/distribution/',
                  heatmap_path='./consensus_plots_postreview/heatmap/',
                  umap_path='./consensus_plots_postreview/umap/',
                  pdb_path='./consensus_plots_postreview/pdb/',
                  heatmap_figure_size=(40,20),
                  vmin=-6,
                  vmax=6,
                  show_plots=False)
#!{pymol_command}


In [ ]:
# Look at specific features, adding domain annotation onto heatmaps
feature_to_heatmap = 'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1'
feature_scores, pymol_command = \
    plot_feature(variant_medians_merged,
                 umap_coords,
                 feature_to_heatmap,
                 "LMNA Tile 3 Nuclear Boundary Abundance",
                 "LMNA.T3.boundaryabundance",
                  dist_plot=True,
                  umap_plot=True,
                  heatmap_plot=True,
                  pdb_plot=True,      
                  highlight_variants=False,
                  save_plots=True,
                  stops=False,
                  threentdels=False,
                  syn_zscore=False,
                  domain_map=LMNA_domain_map,
                  domain_colors=LMNA_domain_colors,
                  anno_height=0.02,
                  other_anno_map=[proline_annotation_dict,ahelical_annotation_dict,interacting_annotation_dict],
                  other_anno_colors=[colorpalette_prolinecluster_dict,colorpalette_positioncluster,colorpalette_interacting],
                  start_pos=start_pos,
                  end_pos=end_pos,
                  wt_nuc_seq=LMNA_WT_nuc_seq,
                  wt_aa_seq=LMNA_WT_aa_seq,
                  remove_nonclassified_variants=False,
                  input_pdb='../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/6jlb.pdb',
                  pymol_view=lmna_pymol_view,
                  image_size=(5000,2500),
                  dpi=1000,
                  pymol_exec="$HOME/pymol/pymol -c",
                  extra_pymol="show sticks\n",
                  dis_path='./consensus_plots_postreview/distribution/',
                  heatmap_path='./consensus_plots_postreview/heatmap/',
                  umap_path='./consensus_plots_postreview/umap/',
                  pdb_path='./consensus_plots_postreview/pdb/',
                  heatmap_figure_size=(40,20),
                  vmin=-6,
                  vmax=6,
                  show_plots=False)
#!{pymol_command}


In [ ]:
# Look at specific features, adding domain annotation onto heatmaps
feature_to_heatmap = 'Mean_NucleiExpanded_Granularity_1_CH1'
feature_scores, pymol_command = \
    plot_feature(variant_medians_merged,
                 umap_coords,
                 feature_to_heatmap,
                 "LMNA Tile 3 Nuclear LMNA Granularity",
                 "LMNA.T3.LMNAGranularity1",
                  dist_plot=True,
                  umap_plot=True,
                  heatmap_plot=True,
                  pdb_plot=True,      
                  highlight_variants=False,
                  save_plots=True,
                  stops=False,
                  threentdels=False,
                  syn_zscore=False,
                  domain_map=LMNA_domain_map,
                  domain_colors=LMNA_domain_colors,
                  anno_height=0.02,
                  other_anno_map=[proline_annotation_dict,ahelical_annotation_dict,interacting_annotation_dict],
                  other_anno_colors=[colorpalette_prolinecluster_dict,colorpalette_positioncluster,colorpalette_interacting],
                  start_pos=start_pos,
                  end_pos=end_pos,
                  wt_nuc_seq=LMNA_WT_nuc_seq,
                  wt_aa_seq=LMNA_WT_aa_seq,
                  remove_nonclassified_variants=False,
                  input_pdb='../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/6jlb.pdb',
                  pymol_view=lmna_pymol_view,
                  image_size=(5000,2500),
                  dpi=1000,
                  pymol_exec="$HOME/pymol/pymol -c",
                  extra_pymol="show sticks\n",
                  dis_path='./consensus_plots_postreview/distribution/',
                  heatmap_path='./consensus_plots_postreview/heatmap/',
                  umap_path='./consensus_plots_postreview/umap/',
                  pdb_path='./consensus_plots_postreview/pdb/',
                  heatmap_figure_size=(40,20),
                  vmin=-6,
                  vmax=6,
                  show_plots=False)
#!{pymol_command}


In [59]:
# Get merged VEP scores
# ----------------------------
# 0) Merged VEP scores
# ----------------------------
lmna_VEP_merged = \
    pd.merge(
        pd.merge(
            lmna_variants_spliceAIfiltered.rename(columns={'protein_variant':'Variant',
                                                           'am_pathogenicity':'AlphaMissense'}),
            lmna_variants_EVE.rename(columns={'EVE_scores_ASM':'EVE'}),
            on='Variant',
            how='inner'
        ),
        lmna_variants_REVEL,
        on='Variant',
        how='inner'
    )

# Plot VEP scores against MIS, landmark features
landmark_features = \
    [
        'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1',
        'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1',
        'Mean_NucleiExpanded_Granularity_1_CH1',
        'Mean_Nuclei_AreaShape_FormFactor',
    ]
features = ['Morphological Impact Score','AUC ROC'] + landmark_features
df_toplot = \
    df_profiles_merged\
        .merge(lmna_VEP_merged,on='Variant',how='inner')\
        [['Variant','EVE','REVEL','EVE_classes_75_pct_retained_ASM','AlphaMissense','UMAP1','UMAP2','Morphological Impact Score']]\
        .merge(variant_medians_merged,on='Variant',how='inner')\
        .merge(roc_auc_lmna, on='Variant', how='left')\
        .merge(clinvar_variants_filterconditions, left_on='Variant', right_on='aaChanges', how='left').query('`Variant Designation` != "VUS"')\
        .assign(Control_Type=lambda df: df['Variant Designation'].map(lambda x: 'PathControl' if x in ['LP','P','P/LP'] else 'Unknown'))
lim_features = {
    'Morphological Impact Score':[0.1,0.9],
    'AUC ROC':[0.4,1],
    'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1':[-10,5],
    'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1':[-10,5],
    'Mean_NucleiExpanded_Granularity_1_CH1':[-5,20],
    'Mean_Nuclei_AreaShape_FormFactor':[-10,10]
}
name_features = {
    'Morphological Impact Score':'Impact\nScore',
    'AUC ROC':'Distinguishability\nScore',
    'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1':'Lamin A Nuclear\nIntensity',
    'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1':'Lamin A Nuclear\nBoundary Intensity',
    'Mean_NucleiExpanded_Granularity_1_CH1':'Lamin A Nuclear\nGranularity 1',
    'Mean_Nuclei_AreaShape_FormFactor':'Nuclear\nCircularity'
}
predictors = ['AlphaMissense', 'EVE', 'REVEL']
colors = {
    'AlphaMissense': '#9A6D38',
    'EVE':           '#33673B',
    'REVEL':         '#CA907E'
}

# Look at variants where there is discordance between AUROC score and VEPs
# ----------------------------
# 1) Make VEP classes
# ----------------------------
am_b = 0.34
am_p = 0.564
revel_b = 0.5
revel_p = 0.75

# define EVE thresholds based on binary
eve_b = lmna_variants_EVE.query('EVE_classes_75_pct_retained_ASM == "Benign"')['EVE_scores_ASM'].max()+10**(-10)
eve_p = lmna_variants_EVE.query('EVE_classes_75_pct_retained_ASM == "Pathogenic"')['EVE_scores_ASM'].min()-10**(-10)

# save thresholds
vep_thresholds = {
    'AlphaMissense' : {"benign":am_b,    "pathogenic":am_p    },
    'EVE'           : {"benign":eve_b,   "pathogenic":eve_p   },
    'REVEL'         : {"benign":revel_b, "pathogenic":revel_p }
}

# What is the name of the distinguishability score column
AUROC_COL  = "AUC ROC"

# Add back synonymous variants
df_toplot2 = pd.concat([
    df_toplot.copy(),
    df_profiles_merged\
        .merge(roc_auc_lmna, on='Variant', how='inner')\
        .query('Variant_Class == "Synonymous"')
        [['Variant','Variant_Class',AUROC_COL]]
])
df_toplot2['AlphaMissense_class'] = df_toplot2['AlphaMissense'].map(lambda x: 'Benign' if x<=am_b    else 'Pathogenic' if x>=am_p    else 'Uncertain')
df_toplot2['REVEL_class']         = df_toplot2['REVEL'].map(lambda x:         'Benign' if x<=revel_b else 'Pathogenic' if x>=revel_p else 'Uncertain')
df_toplot2['EVE_class']           = df_toplot2['EVE_classes_75_pct_retained_ASM']

# ----------------------------
# 2) AUROC binning for DISCORDANCE using YOUR thresholds (independent of histogram bins)
# ----------------------------
# set thresholds based on the synonymous distribution
syn_auroc_scores = df_toplot2.query('Variant_Class == "Synonymous"')[AUROC_COL].values
LOW_THR  = np.nanpercentile(syn_auroc_scores, q=90)
HIGH_THR = np.nanpercentile(syn_auroc_scores, q=99)
print(f"Low threshold: {LOW_THR}\nHigh threshold: {HIGH_THR}")
df_toplot2 = add_auroc_bin_thresholds(df_toplot2, auroc_col=AUROC_COL, low_thr=LOW_THR, high_thr=HIGH_THR, label="AUROC_bin")

# Plot VEP by features, scores, UMAP, histogram over distinguishability using functions in utils
fig = plot_VEPscores_aligned(
    scores=df_toplot2,
    predictors=["AlphaMissense", "EVE", "REVEL"],
    auroc_col=AUROC_COL,
    features=features,
    colors=colors,
    lim_features=lim_features,
    name_features=name_features,
    palette3={"Synon": "darkgreen", "Uncertain": "yellow", "Pathogenic": "red", "Benign": "blue"},
    LOW_THR=LOW_THR,
    HIGH_THR=HIGH_THR,
    show_vep_thresholds=True,
    vep_thresholds=vep_thresholds,
    outpath="./consensus_plots_postreview/LMNAT3.VEPscores.Aligned.pdf",
)
plt.close()

# ----------------------------
# 3) Discordant variants (uses YOUR low/high AUROC thresholds)
# ----------------------------
discordant_variants = find_discordant_variants(
    df_toplot2,
    auroc_col=AUROC_COL,
    auroc_bin_col="AUROC_bin",
    id_col="Variant",
    class_cols=("AlphaMissense_class", "EVE_class", "REVEL_class"),
    keep_mid=False,
)
display(discordant_variants[["Variant", "Predictor", "Discordance", AUROC_COL, "AUROC_bin", "VEP_class"]].drop_duplicates().head(50))


Low threshold: 0.6030476781671579
High threshold: 0.641347267862929


,Variant,Predictor,Discordance,AUC ROC,AUROC_bin,VEP_class
83,D192E,EVE_class,HighAUROC_vs_BenignVEP,0.867067,High,Benign
18,Q255E,AlphaMissense_class,HighAUROC_vs_BenignVEP,0.818332,High,Benign
80,L241M,EVE_class,HighAUROC_vs_BenignVEP,0.785498,High,Benign
115,H252Q,EVE_class,HighAUROC_vs_BenignVEP,0.773347,High,Benign
89,D272G,EVE_class,HighAUROC_vs_BenignVEP,0.769537,High,Benign
113,V256M,EVE_class,HighAUROC_vs_BenignVEP,0.768457,High,Benign
173,Q184E,REVEL_class,HighAUROC_vs_BenignVEP,0.753473,High,Benign
22,Q184E,AlphaMissense_class,HighAUROC_vs_BenignVEP,0.753473,High,Benign
104,Q184E,EVE_class,HighAUROC_vs_BenignVEP,0.753473,High,Benign
10,Q246E,AlphaMissense_class,HighAUROC_vs_BenignVEP,0.748007,High,Benign


In [60]:
# are any discordant variants also clinvar controls?
discordant_variants.merge(clinvar_variants_filterconditions, left_on='Variant', right_on='aaChanges', how='inner').query('`Variant Designation` != "VUS"')


,Variant,AUC ROC,AUROC_bin,VEP_class,Predictor,Discordance,Unnamed: 0,Name,Gene(s),Protein change,...,Oncogenicity date last evaluated,Oncogenicity review status,Unnamed: 24,aaChanges,Mapped Conditions,Variant Designation,DCM,Myopathy,Lipodystrophy,Progeria
0,D230N,0.685651,High,Benign,EVE_class,HighAUROC_vs_BenignVEP,73,NM_170707.4(LMNA):c.688G>A (p.Asp230Asn),LMNA,"D230N, D118N, D149N",...,NaN,NaN,NaN,D230N,FPLD,P/LP,False,False,True,False


In [61]:
# Make table of landmark features + Morphological Impact + AUROC scores for each variant
feature_rename = \
    {
     'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1':'Lamin A Nuclear Intensity',
     'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1':'Lamin A Nuclear Boundary Intensity',
     'Mean_NucleiExpanded_Granularity_1_CH1':'Lamin A Nuclear Granularity 1',
     'Mean_Nuclei_AreaShape_FormFactor': 'Nuclear Shape Factor',
    }
df_supptable_LMNA = \
    variant_medians_merged[['Variant']+landmark_features]\
        .rename(columns={x:feature_rename[x]+' median' for x in landmark_features})
df_supptable_LMNA.columns = \
    [s.replace("\n"," ") for s in df_supptable_LMNA.columns]
df_supptable_LMNA = \
    df_supptable_LMNA\
        .merge(df_profiles_merged[['Variant','Cluster','Variant_Class','Morphological Impact Score',
                                   'UMAP1','UMAP2','NCells_R1','NCells_R2']]\
                   .rename(columns={'Variant_Class':'Variant Type',
                                    'NCells_R1':'Num Cells Replicate 1',
                                    'NCells_R2':'Num Cells Replicate 2'}), 
               on='Variant')\
        .merge(roc_auc_lmna[['Variant','AUC ROC']]\
                   .rename(columns={'AUC ROC':'Distinguishability Score'}), 
               on='Variant', how='left')\
        .merge(df_posanno[['Variant','ProlineCluster','CoilCluster']],
               on='Variant', how='left')
df_supptable_LMNA.to_csv('./SuppTable2.csv', index=False)
